In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.sql.window import Window

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs_v2/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("1. Khoi tao Spark Session cho ALS (Validation Mode)...")
spark = SparkSession.builder \
    .appName("Retrieval_Tuned_ALS_Val") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

Mounted at /content/drive
1. Khoi tao Spark Session cho ALS (Validation Mode)...


In [ ]:
print("2. Doc du lieu va chia khung thoi gian (Validation - 90 ngay)...")
transactions = spark.read.parquet(INPUT_TRANS)
max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)

# Mo rong cua so len 90 ngay de ma tran ALS day dan hon
train_hist_start = val_start - datetime.timedelta(days=90)
test_hist_start = test_start - datetime.timedelta(days=90)

train_hist_df = transactions.filter((F.col("t_dat_date") >= train_hist_start) & (F.col("t_dat_date") < val_start))
test_hist_df = transactions.filter((F.col("t_dat_date") >= test_hist_start) & (F.col("t_dat_date") < test_start))

print(f"Train History range: {train_hist_start} to {val_start}")
print(f"Test History range:  {test_hist_start} to {test_start}")

2. Doc du lieu va chia khung thoi gian (Validation - 90 ngay)...
Train History range: 2020-06-10 to 2020-09-08
Test History range:  2020-06-17 to 2020-09-15


In [ ]:
def generate_als_candidates(history_df, top_n=40):
    # Buoc 1: Chuyen doi ID string sang ID so nguyen (Index) de ALS xu ly
    user_indexer = StringIndexer(inputCol="customer_id", outputCol="user_idx", handleInvalid="keep")
    item_indexer = StringIndexer(inputCol="article_id", outputCol="item_idx", handleInvalid="keep")

    user_model = user_indexer.fit(history_df)
    item_model = item_indexer.fit(history_df)

    df_indexed = user_model.transform(history_df)
    df_indexed = item_model.transform(df_indexed)

    # Buoc 2: Tinh muc do quan tam (Rating) dua tren so lan mua
    ratings = df_indexed.groupBy("user_idx", "item_idx").count().withColumnRenamed("count", "rating")

    # Buoc 3: Huan luyen mo hinh ALS voi tham so da duoc Tuning
    als = ALS(
        maxIter=15, rank=32, regParam=0.05, alpha=40.0,
        userCol="user_idx", itemCol="item_idx", ratingCol="rating",
        coldStartStrategy="drop", implicitPrefs=True, seed=42
    )
    model = als.fit(ratings)

    # Buoc 4: Sinh ung vien cho toan bo khach hang va [TRÍCH XUẤT ĐIỂM SỐ]
    recs = model.recommendForAllUsers(top_n)

    # Bung mang recommendations ra, lay ca item_idx va diem rating (als_score_raw)
    recs_exploded = recs.select("user_idx", F.explode("recommendations").alias("rec")) \
                        .select(
                            "user_idx",
                            F.col("rec.item_idx").alias("item_idx"),
                            F.col("rec.rating").alias("als_score_raw")
                        )

    # Buoc 5: Chuan hoa diem so (Min-Max Scaling) ve muc [0, 1] cho moi khach hang
    window_user = Window.partitionBy("user_idx")
    recs_scaled = recs_exploded.withColumn("max_score", F.max("als_score_raw").over(window_user)) \
                               .withColumn("min_score", F.min("als_score_raw").over(window_user)) \
                               .withColumn("als_score",
                                    F.when(F.col("max_score") == F.col("min_score"), 1.0)
                                     .otherwise((F.col("als_score_raw") - F.col("min_score")) / (F.col("max_score") - F.col("min_score")))
                               )

    # Buoc 6: Map tro lai ID goc cua khach hang va san pham
    user_mapping = df_indexed.select("customer_id", "user_idx").dropDuplicates()
    item_mapping = df_indexed.select("article_id", "item_idx").dropDuplicates()

    # Gop ket qua tra ve: customer_id, article_id, strategy, va diem als_score
    candidates = recs_scaled.join(user_mapping, "user_idx", "inner") \
                            .join(item_mapping, "item_idx", "inner") \
                            .select("customer_id", "article_id", "als_score") \
                            .withColumn("strategy", F.lit("als"))

    return candidates

def evaluate_recall(candidates_df, target_df, target_start, target_end):
    actuals = target_df.filter((F.col("t_dat_date") >= target_start) & (F.col("t_dat_date") < target_end)) \
        .select("customer_id", "article_id").dropDuplicates()

    total_actuals = actuals.count()
    hits = actuals.join(candidates_df, ["customer_id", "article_id"], "inner").dropDuplicates().count()
    recall = hits / total_actuals if total_actuals > 0 else 0

    print(f"   -> Actuals: {total_actuals} | Hits: {hits} | Recall: {recall:.4f}")

In [ ]:
print("\n3. Generating Tuned ALS candidates for Train set (Co diem als_score)...")
train_cands = generate_als_candidates(train_hist_df, top_n=40)
train_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "train_als.parquet")

print("4. Generating Tuned ALS candidates for Test set (Co diem als_score)...")
test_cands = generate_als_candidates(test_hist_df, top_n=40)
test_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "test_als.parquet")

print("5. Evaluating TEST set:")
evaluate_recall(test_cands, transactions, test_start, max_date)
print("Hoan tat! Thu vien ALS (Phien ban nang cap diem so) da san sang.")


3. Generating Tuned ALS candidates for Train set (Co diem als_score)...
4. Generating Tuned ALS candidates for Test set (Co diem als_score)...
5. Evaluating TEST set:
   -> Actuals: 207996 | Hits: 7252 | Recall: 0.0349
Hoan tat! Thu vien ALS (Phien ban nang cap diem so) da san sang.
